# The Audit
### Building Ethical Vision Systems — Ghana Data Science Summit 2026
*Part 2 · Led by Emmanuel Asante*

---

You are not here as a student. You are here as a Quality Assurance Analyst.

A health startup in Accra has built a skin condition detection app for community health workers in rural Ghana. The app uses a computer vision model to classify seven types of skin lesions from phone photos. Before it goes live — before it touches a single patient — we have been asked to evaluate it.

Over the next hour, you will run the model on its benchmark dataset and record what you see. Then you will run it on a different set of images: photos taken on Android phones, outdoors, in West Africa. You will document where it fails and why. And at the end, you will make a recommendation.

The CTO is waiting for your answer. Let's begin.

---

---
### SETUP & ORIENTATION


We start by loading the model and both datasets.

In [ ]:
# Install all dependencies — run this cell once at the start of the session
import subprocess, sys

packages = [
    'torch', 'torchvision', 'timm>=0.9.0',
    'numpy', 'pandas', 'matplotlib', 'seaborn',
    'scikit-learn', 'Pillow', 'tqdm', 'gdown'
]

print('Installing dependencies...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print('WARNINGS/ERRORS:', result.stderr[-2000:])
else:
    print('All dependencies installed.')

In [ ]:
# Standard imports
import os
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
import timm
import gdown
from PIL import Image
from IPython.display import display, Markdown, Image as IPImage
from tqdm.notebook import tqdm

print(f'PyTorch  : {torch.__version__}')
print(f'timm     : {timm.__version__}')
print(f'Platform : {sys.platform}')

In [ ]:
# Configuration — all paths and hyperparameters defined here
import random

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {DEVICE.upper()}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Inference settings ────────────────────────────────────────────────────────
BATCH_SIZE = 32
CONFIDENCE_THRESHOLD = 0.80   # Predictions above this are treated as 'confident'

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('data')
ASSETS_DIR = Path('assets')
ASSETS_DIR.mkdir(exist_ok=True)

# ── Model checkpoint ──────────────────────────────────────────────────────────
# Facilitators: update this path after uploading the .pth file to Google Drive
# or point to a local path if running outside Colab.
MODEL_PATH = '/content/drive/MyDrive/ham10000_efficientnet_b0.pth'

# ── African-context dataset ───────────────────────────────────────────────────
# Facilitators: paste the Google Drive file ID of african_context_dataset.zip here.
# The file ID is the string in: https://drive.google.com/file/d/FILE_ID/view
DRIVE_DATASET_ID = 'https://drive.google.com/drive/folders/1Olk1xU6UujYTVYHpMHXo3NX7e8QoOU3r?usp=sharing'

print(f'Data dir   : {DATA_DIR}')
print(f'Model path : {MODEL_PATH}')
print(f'Config complete.')

In [ ]:
# Mount Google Drive (graceful fallback if running locally)
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    print('Google Drive mounted at /content/drive')
else:
    print('Not running in Colab — skipping Drive mount.')
    print('Ensure MODEL_PATH and DATA_DIR point to local files.')

In [ ]:
import os
utils_dir = '/content/drive/MyDrive/utils'
if '__init__.py' in os.listdir(utils_dir):
    print('`__init__.py` file found in utils directory.')
else:
    print('No `__init__.py` file found in utils directory.')

In [ ]:
# Add repo root to path so utils/ is importable
import sys
from pathlib import Path

REPO_ROOT = Path('/content/drive/MyDrive')   # parent of utils
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import utils
from utils import (
    get_dataloader, get_dataset_info,
    load_audit_model, run_inference, predict_single,
    CLASS_NAMES, CLASS_LABELS,
    compute_metrics, build_failure_table, compute_fairness_metrics,
    plot_confusion_matrix, plot_per_class_f1, plot_metric_comparison,
    plot_confidence_distribution, display_failure_grid,
    plot_options_comparison, plot_fitzpatrick_distribution,
)

print('utils imported successfully')

*⚠ The next cell loads the model. If running on CPU, this takes ~10 seconds.*

In [ ]:
# Load the EfficientNet-B0 model with HAM10000 weights
model = load_audit_model(MODEL_PATH, device=DEVICE)

In [ ]:
# Verify the model — run on one test image to confirm it produces valid output
# Uses a white noise tensor as a sanity check (no real image needed)
torch.manual_seed(RANDOM_SEED)
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)

with torch.no_grad():
    dummy_output = model(dummy_input)
    dummy_probs = torch.softmax(dummy_output, dim=1).squeeze(0).cpu().tolist()

top_idx = int(np.argmax(dummy_probs))
print(f'Output shape    : {dummy_output.shape}  ← (1, 7) as expected')
print(f'Top prediction  : {CLASS_NAMES[top_idx]} ({CLASS_LABELS[CLASS_NAMES[top_idx]]}) — {dummy_probs[top_idx]:.1%}')
print(f'Probabilities sum to : {sum(dummy_probs):.4f}  ← should be 1.0')
print(f'\nModel ready.')

*⚠ The next cell loads both datasets. Expect a few seconds on first load.*

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/data')
print(DATA_DIR)

In [ ]:
# Load both datasets with identical transforms
ham_loader   = get_dataloader('ham10000_test',  batch_size=BATCH_SIZE, data_dir=str(DATA_DIR))
#africa_loader = get_dataloader('african_context', batch_size=BATCH_SIZE, data_dir=str(DATA_DIR))

ham_info    = get_dataset_info('ham10000_test',  data_dir=str(DATA_DIR))
#africa_info = get_dataset_info('african_context', data_dir=str(DATA_DIR))

print(f'HAM10000 test split   : {ham_info["num_samples"]:,} images')
#print(f'African-context dataset: {africa_info["num_samples"]:,} images')
print(f'Both datasets use identical preprocessing transforms.')

Everything is loaded. The model was trained on 10,015 dermatoscopic images collected primarily from clinics in Austria and Australia. It has never seen a patient from Ghana. Let's find out what that means.

---

---
### THE BASELINE RUN


The model has strong benchmark numbers. The startup's CTO referenced an 87% accuracy figure in the pitch deck and wants to launch before the end of the quarter. Our first job is to reproduce those numbers — run the model on the same held-out test set it was evaluated on, and record exactly what we see.

*⚠ Running inference on ~2,000 images. On CPU this takes 3–5 minutes. On a T4 GPU, under 1 minute.*

In [ ]:
# Run inference on the HAM10000 test split and record predictions
print('Running model on HAM10000 test split...')
ham_results = run_inference(model, ham_loader, DEVICE)

print(f'\nInference complete.')
print(f'  Total images processed : {len(ham_results["predictions"]):,}')
print(f'  First prediction       : {CLASS_LABELS[CLASS_NAMES[ham_results["predictions"][0]]]}')

In [ ]:
# Compute and display baseline performance metrics
ham_metrics = compute_metrics(
    ham_results['predictions'],
    ham_results['true_labels'],
    ham_results['probabilities']
)

print('=' * 50)
print('  HAM10000 TEST SET — PERFORMANCE SUMMARY')
print('=' * 50)
print(f'  Overall accuracy   : {ham_metrics["overall_accuracy"]:.1%}')
print(f'  Macro F1 score     : {ham_metrics["macro_f1"]:.3f}')
print(f'  Weighted F1 score  : {ham_metrics["weighted_f1"]:.3f}')
print()
print('  Per-class accuracy:')
for cls, acc in sorted(ham_metrics['per_class_accuracy'].items(), key=lambda x: -x[1]):
    label = CLASS_LABELS[cls].ljust(24)
    bar = '█' * int(acc * 20)
    print(f'    {label}  {acc:.1%}  {bar}')
print('=' * 50)

In [ ]:
# Plot the normalised confusion matrix
fig = plot_confusion_matrix(
    ham_results['true_labels'],
    ham_results['predictions'],
    title='Confusion Matrix — HAM10000 Test Split',
    save=True
)
plt.show()

In [ ]:
# Plot per-class F1 bar chart (sorted descending)
fig = plot_per_class_f1(
    ham_metrics,
    title='Per-Class F1 — HAM10000 Test Split',
    save=True
)
plt.show()

In [ ]:
# Display the top 5 most confident CORRECT predictions
# These are the cases where the model is most certain and right.
import operator

preds     = ham_results['predictions']
labels    = ham_results['true_labels']
probs_all = ham_results['probabilities']
ids       = ham_results['image_ids']

# Find correct predictions sorted by max confidence
correct_with_conf = [
    (max(probs_all[i]), i)
    for i in range(len(preds))
    if preds[i] == labels[i]
]
correct_with_conf.sort(reverse=True)
top5_correct_idx = [idx for _, idx in correct_with_conf[:5]]

ham_images_dir = DATA_DIR / 'ham10000_test' / 'images'

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
fig.suptitle('Top 5 Most Confident Correct Predictions — HAM10000', fontsize=12, fontweight='bold', color='#1A2535')

for ax, idx in zip(axes, top5_correct_idx):
    img_path = ham_images_dir / f'{ids[idx]}.jpg'
    if img_path.exists():
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
    else:
        ax.set_facecolor('#F3F4F6')
    ax.axis('off')
    conf = max(probs_all[idx])
    cls = CLASS_NAMES[preds[idx]]
    ax.set_title(f'{CLASS_LABELS[cls]}\n{conf:.1%}', fontsize=7.5, color='#028090')

plt.tight_layout()
plt.show()

In [ ]:
# Display the top 5 most confident WRONG predictions
# The model is highly certain — and completely incorrect.
# This is the first hint that confidence does not mean accuracy.

wrong_with_conf = [
    (max(probs_all[i]), i)
    for i in range(len(preds))
    if preds[i] != labels[i]
]
wrong_with_conf.sort(reverse=True)
top5_wrong_idx = [idx for _, idx in wrong_with_conf[:5]]

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
fig.suptitle('⚠ Top 5 Most Confident WRONG Predictions — HAM10000', fontsize=12, fontweight='bold', color='#DC2626')

for ax, idx in zip(axes, top5_wrong_idx):
    img_path = ham_images_dir / f'{ids[idx]}.jpg'
    if img_path.exists():
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
    else:
        ax.set_facecolor('#F3F4F6')
    ax.axis('off')
    conf = max(probs_all[idx])
    pred_cls = CLASS_NAMES[preds[idx]]
    true_cls = CLASS_NAMES[labels[idx]]
    ax.set_title(
        f'True: {CLASS_LABELS[true_cls]}\nPred: {CLASS_LABELS[pred_cls]} ({conf:.1%})',
        fontsize=7, color='#DC2626'
    )

plt.tight_layout()
plt.show()

print(f'\nThe model was wrong on these images with confidence as high as {max(c for c, _ in wrong_with_conf):.1%}.')
print('High confidence is not the same as high accuracy.')

---

#### ✍️ Reflection 2.1 — First Impressions

The model reports **40.6% overall accuracy** on the HAM10000 test set.

Based only on what you see in the metrics above, would you recommend this model for deployment across rural clinics in Ghana?

| Your decision | Your one-sentence justification |
|---|---|
| ☐ Yes / ☐ No | *Write here* |

> Keep your answer. You will revisit it at the end of the lab.

---

---
###  THE STRESS TEST

Same model. Same code. Different images.

The next dataset was collected from phones — Android devices, outdoors, under natural West African light. No dermatoscope. No controlled clinic environment. These are the exact conditions a health worker in Brong-Ahafo would be operating under. The model has never seen images like this. Run it.

In [ ]:
# Run inference on the African-context stress-test dataset
print('Running model on African-context dataset...')
africa_results = run_inference(model, africa_loader, DEVICE)

print(f'\nInference complete.')
print(f'  Total images processed : {len(africa_results["predictions"]):,}')

In [ ]:
# Compute the same metrics on the African-context dataset
africa_metrics = compute_metrics(
    africa_results['predictions'],
    africa_results['true_labels'],
    africa_results['probabilities']
)

print('=' * 50)
print('  AFRICAN-CONTEXT DATASET — PERFORMANCE SUMMARY')
print('=' * 50)
print(f'  Overall accuracy   : {africa_metrics["overall_accuracy"]:.1%}')
print(f'  Macro F1 score     : {africa_metrics["macro_f1"]:.3f}')
print(f'  Weighted F1 score  : {africa_metrics["weighted_f1"]:.3f}')
print()
print('  Per-class accuracy:')
for cls, acc in sorted(africa_metrics['per_class_accuracy'].items(), key=lambda x: -x[1]):
    label = CLASS_LABELS[cls].ljust(24)
    bar = '█' * int(acc * 20) if not np.isnan(acc) else '—'
    acc_str = f'{acc:.1%}' if not np.isnan(acc) else 'N/A'
    print(f'    {label}  {acc_str}  {bar}')
print('=' * 50)

In [ ]:
# Side-by-side comparison — this is the central visual of the lab
# The gap between the two datasets must be immediately legible.
fig = plot_metric_comparison(
    ham_metrics,
    africa_metrics,
    label_a='HAM10000 Test',
    label_b='African Context',
    save=True
)
plt.show()

# Print the headline gap numbers
acc_gap = ham_metrics['overall_accuracy'] - africa_metrics['overall_accuracy']
f1_gap  = ham_metrics['macro_f1'] - africa_metrics['macro_f1']
print(f'Overall accuracy gap : {acc_gap:+.1%}  (HAM10000 - African Context)')
print(f'Macro F1 gap         : {f1_gap:+.3f}')

In [ ]:
# Display the 8 worst failures from the African-context dataset
# Sorted by confidence descending — these are the cases where the model
# was most certain and most wrong.

africa_preds  = africa_results['predictions']
africa_labels = africa_results['true_labels']
africa_probs  = africa_results['probabilities']
africa_ids    = africa_results['image_ids']

africa_images_dir = DATA_DIR / 'african_context' / 'images'

# Get Fitzpatrick types if available
try:
    africa_labels_df = pd.read_csv(DATA_DIR / 'african_context' / 'african_context_labels.csv')
    fitz_map = dict(zip(africa_labels_df['image_id'], africa_labels_df.get('fitzpatrick_type', [None]*len(africa_labels_df))))
except Exception:
    fitz_map = {}

# Find wrong predictions sorted by confidence
wrong_cases = [
    (max(africa_probs[i]), i)
    for i in range(len(africa_preds))
    if africa_preds[i] != africa_labels[i]
]
wrong_cases.sort(reverse=True)
worst8_idx = [idx for _, idx in wrong_cases[:8]]

# Load images
worst8_images, worst8_preds, worst8_true, worst8_conf, worst8_fitz = [], [], [], [], []
for idx in worst8_idx:
    img_path = africa_images_dir / f'{africa_ids[idx]}.jpg'
    if img_path.exists():
        worst8_images.append(Image.open(img_path).convert('RGB'))
    else:
        worst8_images.append(Image.new('RGB', (224, 224), color=(200, 200, 200)))
    worst8_preds.append(CLASS_LABELS[CLASS_NAMES[africa_preds[idx]]])
    worst8_true.append(CLASS_LABELS[CLASS_NAMES[africa_labels[idx]]])
    worst8_conf.append(max(africa_probs[idx]))
    worst8_fitz.append(fitz_map.get(africa_ids[idx]))

fig = display_failure_grid(
    worst8_images, worst8_preds, worst8_true, worst8_conf,
    fitzpatrick_types=worst8_fitz,
    n=8,
    title='8 Worst Failures — African-Context Dataset',
    save=True
)
plt.show()

In [ ]:
# Overlaid confidence distribution — HAM10000 vs African Context
# Note: the model is similarly confident on both datasets despite being
# far less accurate on the African-context set.
fig = plot_confidence_distribution(
    ham_results['probabilities'],
    africa_results['probabilities'],
    label_a='HAM10000 Test',
    label_b='African Context',
    save=True
)
plt.show()

ham_mean_conf    = np.mean([max(p) for p in ham_results['probabilities']])
africa_mean_conf = np.mean([max(p) for p in africa_results['probabilities']])
print(f'Mean max confidence — HAM10000       : {ham_mean_conf:.1%}')
print(f'Mean max confidence — African context : {africa_mean_conf:.1%}')
print(f'\nThe model is equally confident on both datasets.')
print(f'Accuracy on HAM10000       : {ham_metrics["overall_accuracy"]:.1%}')
print(f'Accuracy on African context : {africa_metrics["overall_accuracy"]:.1%}')

---

#### ✍️ Reflection 3.1 — The Performance Gap

Complete the table with what you observe from the metrics comparison:

| Metric | HAM10000 test | African-context | Difference |
|---|---|---|---|
| Overall accuracy | | | |
| Worst-performing class | | | |
| Best-performing class | | | |
| Average confidence on wrong predictions | | | |

**In 2–3 sentences:** Is the pattern of failures random, or do you see a systematic trend?

*Write here*

---

#### ✍️ Reflection 3.2 — Which Errors Matter Most?

Your failure table shows three error types. Rank them by clinical severity for this use case:

| Error type | Count | Your severity ranking (1 = most dangerous) | Why |
|---|---|---|---|
| False negative (condition missed) | | | |
| False positive (condition flagged incorrectly) | | | |
| High-confidence wrong prediction | | | |

> A false negative in melanoma detection means a patient goes home unscreened. A false positive means an unnecessary referral. Neither is neutral.

---

---
###  DOCUMENT FAILURES & ROOT CAUSE


We have seen that the model performs worse on African-context images. Two questions remain: what exactly broke, and why? We answer them in order. Part A is about documentation — building a structured record of failures. Part B is about root cause — tracing the failure to its origin in the pipeline.

**Part A — Document the Failures**

In [ ]:
# Build the failure table — one row per image, classified by error type
failure_table = build_failure_table(
    africa_results['predictions'],
    africa_results['true_labels'],
    africa_results['probabilities'],
    africa_results['image_ids'],
    confidence_threshold=CONFIDENCE_THRESHOLD
)

print(f'Total images in failure table: {len(failure_table)}')
print(f'\nError type breakdown:')
print(failure_table['error_type'].value_counts().to_string())
print()

# Display the table sorted by confidence descending
display(failure_table.style.format({'confidence': '{:.1%}'})
        .background_gradient(subset=['confidence'], cmap='RdYlGn'))

In [ ]:
# YOUR CODE HERE
# Filter failure_table to show only high-confidence wrong predictions
# (where confidence > 0.8 AND error_type != 'correct')
# Hint: you can do this in one line of pandas filtering.

high_conf_wrong = # your code here

print(f'High-confidence wrong predictions (confidence > {CONFIDENCE_THRESHOLD:.0%}): {len(high_conf_wrong)}')
display(high_conf_wrong.style.format({'confidence': '{:.1%}'}))

In [ ]:
# Display error type summary as a styled table
error_summary = (
    failure_table[failure_table['error_type'] != 'correct']
    .groupby('error_type')
    .agg(
        count=('error_type', 'count'),
        mean_confidence=('confidence', 'mean'),
        max_confidence=('confidence', 'max')
    )
    .reset_index()
    .sort_values('count', ascending=False)
)

display(error_summary.style
    .format({'mean_confidence': '{:.1%}', 'max_confidence': '{:.1%}'})
    .set_caption('Failure Summary by Error Type — African-Context Dataset'))

**Part B — Root Cause**

The failure is documented. Now we trace it. The model did not fail because of a bug or a misconfigured threshold. It failed because of what it was trained on. Read the datasheet below — it tells you exactly where the training data came from.

In [ ]:
# Display the HAM10000 datasheet inline
datasheet_path = Path('data/ham10000_datasheet.md')
with open(datasheet_path, 'r', encoding='utf-8') as f:
    datasheet_content = f.read()

display(Markdown(datasheet_content))

In [ ]:
# Fitzpatrick skin type distribution for HAM10000
# These are estimated values inferred from collection geography — not measured.
# The original dataset authors did not record Fitzpatrick types.
HAM10000_FITZPATRICK_ESTIMATED = {
    1: 2500,   # Type I  ~25%
    2: 3500,   # Type II ~35%
    3: 2500,   # Type III ~25%
    4: 1000,   # Type IV ~10%
    5: 400,    # Type V  ~4%
    6: 115,    # Type VI <1%
}

fig = plot_fitzpatrick_distribution(
    HAM10000_FITZPATRICK_ESTIMATED,
    dataset_name='HAM10000 (estimated)',
    save=True
)
plt.show()

total = sum(HAM10000_FITZPATRICK_ESTIMATED.values())
types_v_vi = HAM10000_FITZPATRICK_ESTIMATED[5] + HAM10000_FITZPATRICK_ESTIMATED[6]
print(f'Estimated Type V–VI representation: {types_v_vi/total:.1%} of the training set.')

In [ ]:
# Side-by-side: HAM10000 image (dermatoscope) vs African-context image (phone camera)
# This makes the acquisition quality difference visible at a glance.

ham_sample_paths = sorted(list((DATA_DIR / 'ham10000_test' / 'images').glob('*.jpg')))[:3]
africa_sample_paths = sorted(list((DATA_DIR / 'african_context' / 'images').glob('*.jpg')))[:3]

if ham_sample_paths and africa_sample_paths:
    fig, axes = plt.subplots(2, 3, figsize=(10, 6))
    fig.suptitle('Acquisition Quality: Dermatoscope (top) vs Phone Camera (bottom)',
                 fontsize=12, fontweight='bold', color='#1A2535')

    for i, path in enumerate(ham_sample_paths[:3]):
        axes[0, i].imshow(Image.open(path).convert('RGB'))
        axes[0, i].axis('off')
        axes[0, i].set_title('HAM10000\n(Dermatoscope)', fontsize=8, color='#028090')

    for i, path in enumerate(africa_sample_paths[:3]):
        axes[1, i].imshow(Image.open(path).convert('RGB'))
        axes[1, i].axis('off')
        axes[1, i].set_title('African Context\n(Phone camera)', fontsize=8, color='#D97706')

    plt.tight_layout()
    plt.show()
else:
    print('Sample images not found — ensure both datasets are downloaded.')

---

#### ✍️ Reflection 4.1 — Tracing the Failure

You have now read the HAM10000 datasheet. Answer the following:

**1. Where was the HAM10000 data collected?**
*Write here*

**2. What Fitzpatrick skin types are represented, and in what proportions?**
*Write here*

**3. Is this a model problem or a data problem? Explain your reasoning.**
*Write here*

**4. At what point in the pipeline did this failure originate — preprocessing, labelling, training data, or model architecture?**
*Write here*

---

---
###  PROPOSE A PATH


Analysis is not enough. An auditor's job is to make a recommendation. There is no single right answer here — but every answer has consequences, and some of those consequences fall on people who are not in this room.

⚖️ **Three paths forward:**

**Option A — Collect and fine-tune**
Gather 500–1,000 images of skin conditions from African patients, label them with local dermatologists, and fine-tune the model before deployment. The model learns what it's actually going to see.

**Option B — Restricted deployment with human review**
Deploy immediately with a confidence threshold: any prediction below 80% triggers mandatory review by a supervising clinician before the result is shown to the health worker. No deployment in regions without clinician backup within a defined response window.

**Option C — Halt deployment**
The data gap is too large to bridge with a threshold. Recommend the startup pause deployment until a representative dataset has been assembled and the model retrained from scratch on data that reflects the actual patient population.

In [ ]:
# Cost-benefit comparison across the three options
fig = plot_options_comparison(save=True)
plt.show()

---

**Real-world precedents:**

**Option A — The SCIN template.** In 2024, Google Research released the Skin Condition Image Network (SCIN), a dataset built specifically to address the demographic gap in dermatology AI. The collection process involved partnering with community health organisations, compensating contributors, and working with board-certified dermatologists to label images across all Fitzpatrick types. The result was a dataset explicitly designed to represent patients who are typically excluded from medical AI training sets. Option A follows this same logic at a smaller scale.

**Option B — The pulse oximeter precedent.** Following multiple studies published in 2020–2021 showing that pulse oximeters systematically overestimate blood oxygen levels in patients with darker skin — leading to delayed COVID-19 treatment for Black patients — several health bodies updated clinical guidance to require clinician verification before acting on pulse oximeter readings for darker-skinned patients. Option B mirrors this approach: don't remove the tool, but mandate human oversight where it is known to be less reliable.

**Option C — FDA rejection precedent.** In 2023, the FDA declined to approve an AI diagnostic tool on the grounds that its training data did not include sufficient demographic diversity to support claims of generalised clinical performance. The agency's position was that demographic gaps in training data are a safety issue, not a future improvement to be addressed post-deployment. Option C reflects this position: if the data gap is large enough, deployment itself is the harm.

---

#### ✍️ Reflection 5.1 — Your Recommendation

Select your recommended path and defend it in 3–5 sentences:

**Option A — Collect and fine-tune**
Gather 500–1,000 images of African skin conditions, label them with local dermatologists, and fine-tune the model before deployment.

**Option B — Restricted deployment with human review**
Deploy with a confidence threshold: predictions below 80% trigger mandatory human review by a supervising clinician. No deployment in regions without clinician backup.

**Option C — Halt deployment**
The data gap is too large. Recommend the startup pause deployment until a representative dataset is assembled and the model is retrained from scratch.

| Your choice | Option A / Option B / Option C |
|---|---|
| **Primary reason** | *Write here* |
| **Biggest risk of your choice** | *Write here* |
| **Who bears the cost if you are wrong?** | *Write here* |

---

#### ✍️ Reflection 5.2 — Looking Back

In Reflection 2.1, you answered: *[your answer from earlier]*.

**Has your answer changed?**

| Same / Changed | What changed your thinking (or didn't)? |
|---|---|
| | *Write here* |

---

---

The startup's CTO is waiting for your answer. The health workers in Brong-Ahafo are waiting for the app. These are not abstractions.

The model you just audited is a real class of system being deployed across Sub-Saharan Africa right now — sometimes with this level of scrutiny, often without it. What you did in the last hour is the work.

---
*Ghana Data Science Summit 2026 · Building Ethical Vision Systems*
*Tutorial leads: Nana Sam Yeboah (Part 1) · Emmanuel Asante (Part 2)*